# Optimal trees and rule lists

A decision tree or a rule list is the oldest readable model there is: a handful of threshold questions, answered in order. Ordinary tree learners (CART, and the trees inside the random forest) grow greedily, one best split at a time, so a small tree is rarely the *best* small tree. GOSDT (Lin, Zhong, Hu, Rudin & Seltzer 2020) and CORELS (Angelino, Larus-Stone, Alabi, Seltzer & Rudin 2017) instead search for the tree or rule list that is provably optimal for a declared trade-off between training error and size, using branch-and-bound over binarized features. The question for this notebook: **how many bits does a provably best small tree or rule list carry, and what does it ask?**

Both solvers are binary classifiers. The tiers notebook encoded the four tiers in two **Gray-code bits** — tier 0 = 00, 1 = 01, 2 = 11, 3 = 10 — so a one-tier error flips exactly one bit. We fit one tree (or rule list) per bit: the high bit says "tier 2 or 3" (soluble side), the low bit says "tier 1 or 2" (the middle). Each leaf or rule carries the training frequency of its bit, so the model's answer for a compound is two probabilities, multiplied into a tier distribution. The multiplication assumes the two bits are independent given the leaf; that assumption is tested below with the forest before the trees are read.

## Framework

- **Data and splits.** As in the other notebooks: unflagged votes with weight $1/n$, random and cluster 10-fold cross-validation, five learned splits; the RDKit descriptor set the features notebook carried downstream. Neither solver takes sample weights, so each compound is **one training row labelled with its consensus tier** (the tier of the median of its unflagged votes); the predictions are then scored on every vote as everywhere else.
- **Binarization, declared.** The solvers need yes/no features. Thresholds come from GOSDT's threshold guessing: a gradient-boosting model of 40 depth-1 stumps is fitted on the train side for the bit in question, and each stump's split becomes one binary feature "descriptor ≤ value" (the direction is asserted in the driver, not assumed) — at most 40 per bit, fitted on train only. Both solvers see the same binary features.
- **Solver settings, declared.** GOSDT: depth at most 5, 60 s per search, regularization $\lambda_\text{reg}$ from the grid {0.005, 0.01, 0.02} (the training-accuracy cost of one more leaf). CORELS: rules with at most two conditions, the same grid for its per-rule penalty, one million search iterations at most. The regularization is chosen per solver on the validation sides of the random splits; the same value serves both bits.
- **Probabilities.** Each leaf (rule) gives the training share of compounds with the bit set, with +0.5 smoothing. Tier probabilities are the product of the two bits' probabilities, then shrunk by $\lambda$ chosen on validation as for every other model.
- **Reference.** The random forest on the same descriptors, as a four-class model and as two per-bit forests multiplied — the latter isolates what the two-bit factorization costs.
- **Scores.** From `scripts/aqsoldb_harness.py`: information $= H(T) - \text{log-loss}$, hard-tier accuracies; two test populations (benchmark 0.963 bits / 76.5%).
- **Description length, declared.** For a tree: leaves, depth, descriptors used; for a rule list: rules and conditions. The series-wide count — numbers needed to write the model down — is, per tree, two numbers per internal node (a descriptor and a threshold) and one per leaf (a probability), summed over the two trees; per rule list, two numbers per condition and one per rule.
- **Environment.** The solvers live in a Python 3.9 side environment (`tools/trees_and_rules/`, built by `setup.sh`); this notebook runs `tools/trees_and_rules/run.py` there and reads back the per-compound probabilities and the models as text. Nothing from that environment is imported here.

In [1]:
import itertools
import json
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "scripts"))
from aqsoldb_harness import METRICS, calibrate, corrected_se, formatted, score  # noqa: E402

votes = pd.read_csv(ROOT / "data" / "aqsoldb_tiers.csv")
values = votes[votes["weight"] > 0].reset_index(drop=True)
TIERS = int(values["tier"].max()) + 1
splits = pd.read_csv(ROOT / "data" / "aqsoldb_splits.csv", index_col="InChIKey")
learned = pd.read_csv(ROOT / "data" / "aqsoldb_splits_ls.csv", index_col="InChIKey").reindex(splits.index)
features = pd.read_csv(ROOT / "data" / "aqsoldb_features.csv", index_col="InChIKey").reindex(splits.index)
CV_SPLITS = [c for c in splits.columns if c.startswith("cv_") and c[3:].isdigit()]
CLUSTER_SPLITS = [c for c in splits.columns if c.startswith("cluster_") and c[8:].isdigit()]
LS_SPLITS = [c for c in learned.columns if c.startswith("ls_") and c[3:].isdigit()]
SCHEMES = {"random cross-validation": CV_SPLITS, "cluster cross-validation": CLUSTER_SPLITS, "learning to split": LS_SPLITS}
all_splits = splits.join(learned[LS_SPLITS])

compound = splits.index.get_indexer(values["InChIKey"])
assert (compound >= 0).all()
y, w = values["tier"].to_numpy(), values["weight"].to_numpy()
n_compounds = len(splits)
repeated = np.bincount(compound, minlength=n_compounds) > 1
consensus = values.groupby("InChIKey")["consensus_tier"].first().reindex(splits.index).to_numpy().astype(int)
X = features.to_numpy()
GRAY = {"bit_1": (2, 3), "bit_2": (1, 2)}  # tiers in which each bit is 1
BITS = {b: np.isin(consensus, t).astype(int) for b, t in GRAY.items()}
TIER_BITS = np.array([[int(t in GRAY["bit_1"]), int(t in GRAY["bit_2"])] for t in range(TIERS)])  # tier -> (bit_1, bit_2)
assert len({tuple(r) for r in TIER_BITS}) == TIERS, "the Gray code must be one-to-one"

SIDE_PYTHON = ROOT / "tools" / "trees_and_rules" / ".venv" / "bin" / "python"
DRIVER = ROOT / "tools" / "trees_and_rules" / "run.py"
OUT = ROOT / "data" / "trees_and_rules"
REGULARIZATION = [0.005, 0.01, 0.02]
assert SIDE_PYTHON.exists(), "build the side environment first: tools/trees_and_rules/setup.sh"


def calibrate_on(proba, on):
    r = on[compound]
    return calibrate(y[r], proba[compound[r]], w[r])


def evaluate(proba, on, lam):
    r = on[compound]
    return score(y[r], proba[compound[r]], w[r], lam)


def sides(split):
    side = all_splits[split].to_numpy()
    train, validation, test = side == "train", side == "validation", side == "test"
    assert train.any() and validation.any() and test.any(), f"{split}: a side has no rows"
    return train, validation, test


def test_to_train(split_list):
    return (all_splits[split_list[0]] == "test").sum() / (all_splits[split_list[0]] == "train").sum()


def validation_to_train(split_list):
    return (all_splits[split_list[0]] == "validation").sum() / (all_splits[split_list[0]] == "train").sum()


def from_bits(p1: np.ndarray, p2: np.ndarray) -> np.ndarray:
    """Tier distribution per compound from the two bit probabilities, assuming independence."""
    p = np.stack([np.where(TIER_BITS[t, 0], p1, 1 - p1) * np.where(TIER_BITS[t, 1], p2, 1 - p2) for t in range(TIERS)], axis=1)
    return p / p.sum(axis=1, keepdims=True)


def run_driver(split_list, regularization):
    """Fit both solvers in the side environment for these splits and regularizations; results land in OUT."""
    started = time.time()
    cmd = [str(SIDE_PYTHON), str(DRIVER), "--root", str(ROOT), "--splits", *split_list, "--regularization", *map(str, regularization), "--out", str(OUT)]
    done = subprocess.run(cmd, capture_output=True, text=True)
    print(done.stdout[-1500:])
    assert done.returncode == 0, done.stderr[-3000:]
    print(f"{len(split_list)} splits × {len(regularization)} regularizations in {(time.time() - started) / 60:.0f} min")


def load_driver():
    preds = {m: pd.read_csv(OUT / f"predictions_{m}.csv", index_col="InChIKey").reindex(splits.index) for m in ("gosdt", "corels")}
    return preds, json.load(open(OUT / "models.json"))


print(f"{len(values)} votes, {n_compounds} compounds, {X.shape[1]} descriptors; consensus tiers: {np.bincount(consensus)}")

12013 votes, 9946 compounds, 152 descriptors; consensus tiers: [1583 2234 3165 2964]


## Does the two-bit factorization lose information?

Before any tree is fitted: the random forest as a four-class model against two per-bit forests multiplied, on the validation sides of the random splits. If the product falls well short of the four-class forest, the factorization itself — not the solvers — would cap what the trees can reach.

In [2]:
RF_SETTINGS = dict(n_estimators=500, n_jobs=-1, random_state=0)


def forest_4class(train):
    rows = train[compound]
    model = RandomForestClassifier(**RF_SETTINGS).fit(X[compound[rows]], y[rows], sample_weight=w[rows])
    assert list(model.classes_) == list(range(TIERS))
    return model, model.predict_proba(X)


def forest_bits(train):
    """Two per-bit forests on the consensus-tier bits (one row per compound, as the solvers see the data)."""
    p = {}
    for bit, label in BITS.items():
        model = RandomForestClassifier(**RF_SETTINGS).fit(X[train], label[train])
        p[bit] = model.predict_proba(X)[:, list(model.classes_).index(1)]
    return from_bits(p["bit_1"], p["bit_2"])


forest_probas, rows = {}, []
for split in CV_SPLITS:
    train, validation, test = sides(split)
    _, four = forest_4class(train)
    product = forest_bits(train)
    forest_probas["4-class", split], forest_probas["bits", split] = four, product
    for name, proba in [("random forest, 4 classes", four), ("random forest, 2 Gray bits multiplied", product)]:
        lam = calibrate_on(proba, validation)
        rows.append({"model": name, "split": split, **evaluate(proba, validation, lam)})
check = pd.DataFrame(rows).groupby("model", sort=False)["information (bits)"].agg(["mean", "var"])
check["SE"] = np.sqrt((1 / len(CV_SPLITS) + validation_to_train(CV_SPLITS)) * check.pop("var"))
check.rename(columns={"mean": "validation information (bits)"}).round(3)

,validation information (bits),SE
model,,
"random forest, 4 classes",1.009,0.009
"random forest, 2 Gray bits multiplied",0.992,0.009


## Fitting the solvers on the random splits

The driver runs both solvers for every random split, bit and regularization. The table reports, per solver and regularization, the validation information of the multiplied bits (mean ± SE over the ten splits) and the mean size; the regularization with the best validation information is chosen per solver.

In [3]:
run_driver(CV_SPLITS, REGULARIZATION)
preds, models = load_driver()


def proba_of(method, split, reg):
    return from_bits(preds[method][f"{split}|bit_1|{reg}"].to_numpy(), preds[method][f"{split}|bit_2|{reg}"].to_numpy())


def size_of(method, split, reg) -> dict:
    m1, m2 = models[f"{method}|{split}|bit_1|{reg}"], models[f"{method}|{split}|bit_2|{reg}"]
    if method == "gosdt":
        return {"leaves (both trees)": m1["leaves"] + m2["leaves"], "max depth": max(m1["depth"], m2["depth"]),
                "descriptors used": m1["features used"] + m2["features used"],
                "numbers to write it down": sum(2 * (m["leaves"] - 1) + m["leaves"] for m in (m1, m2))}
    return {"rules (both lists)": m1["rules"] + m2["rules"], "conditions": m1["antecedents"] + m2["antecedents"],
            "numbers to write it down": sum(2 * m["antecedents"] + m["rules"] for m in (m1, m2))}


rows = []
for method, reg, split in itertools.product(("gosdt", "corels"), REGULARIZATION, CV_SPLITS):
    train, validation, test = sides(split)
    proba = proba_of(method, split, reg)
    lam = calibrate_on(proba, validation)
    rows.append({"solver": method, "regularization": reg, "split": split, "λ": lam, **evaluate(proba, validation, lam), **size_of(method, split, reg)})
sweep_rows = pd.DataFrame(rows)
by = sweep_rows.groupby(["solver", "regularization"], sort=False)
sweep = by["information (bits)"].agg(["mean", "var"])
sweep["SE"] = np.sqrt((1 / len(CV_SPLITS) + validation_to_train(CV_SPLITS)) * sweep.pop("var"))
sweep = sweep.rename(columns={"mean": "validation information (bits)"})
for col in ["accuracy: same tier (%)", "leaves (both trees)", "rules (both lists)", "numbers to write it down"]:
    sweep[col] = by[col].mean()
BEST = {m: float(sweep.loc[m, "validation information (bits)"].idxmax()) for m in ("gosdt", "corels")}
print("chosen regularization:", BEST)
sweep.round(3)

  gosdt  2 leaves   0.0s | corels  3 rules   1.0s | 663s
cv_7|bit_1|0.01        gosdt  2 leaves   0.0s | corels  2 rules   0.0s | 663s
cv_7|bit_1|0.02        gosdt  2 leaves   0.0s | corels  2 rules   0.0s | 663s
cv_7|bit_2|0.005       gosdt  6 leaves   5.3s | corels  3 rules   1.6s | 671s
cv_7|bit_2|0.01        gosdt  4 leaves   2.9s | corels  3 rules   1.8s | 676s
cv_7|bit_2|0.02        gosdt  4 leaves   0.8s | corels  3 rules  53.6s | 730s
cv_8|bit_1|0.005       gosdt  2 leaves   4.7s | corels  3 rules   1.5s | 738s
cv_8|bit_1|0.01        gosdt  2 leaves   0.1s | corels  2 rules   2.1s | 740s
cv_8|bit_1|0.02        gosdt  2 leaves   0.0s | corels  2 rules   0.5s | 741s
cv_8|bit_2|0.005       gosdt  6 leaves   4.2s | corels  3 rules   1.6s | 748s
cv_8|bit_2|0.01        gosdt  4 leaves   2.0s | corels  3 rules   1.7s | 751s
cv_8|bit_2|0.02        gosdt  4 leaves   0.4s | corels  3 rules  52.6s | 804s
cv_9|bit_1|0.005       gosdt  2 leaves   0.0s | corels  2 rules   0.0s | 806s
cv_9|bi

validation information (bits)     SE  \
solver regularization                                         
gosdt  0.005                                   0.556  0.007   
       0.010                                   0.553  0.005   
       0.020                                   0.533  0.015   
corels 0.005                                   0.549  0.011   
       0.010                                   0.535  0.006   
       0.020                                   0.537  0.006   

                       accuracy: same tier (%)  leaves (both trees)  \
solver regularization                                                 
gosdt  0.005                            64.054                  7.7   
       0.010                            63.515                  6.9   
       0.020                            62.684                  6.1   
corels 0.005                            63.658                  NaN   
       0.010                            63.371                  NaN   
       0.020                            63.384                  NaN   

                       rules (both lists)  numbers to write it down  
solver regularization                                                
gosdt  0.005                          NaN                      19.1  
       0.010                          NaN                      16.7  
       0.020                          NaN                      14.3  
corels 0.005                          5.9                      21.3  
       0.010                          5.1                      16.9  
       0.020                          5.0                      16.2

## Against the forest, on three kinds of split

The chosen regularization per solver, now also fitted on the cluster and learned splits; the four-class forest alongside. Each model is calibrated on its split's validation side and scored on its test side.

In [4]:
run_driver(CLUSTER_SPLITS + LS_SPLITS, sorted(set(BEST.values())))
preds, models = load_driver()
SUBSETS = {"all test compounds": np.ones(n_compounds, bool), "test compounds with 2+ unflagged votes": repeated}
NAMES = {"gosdt": f"GOSDT trees (λ_reg = {BEST['gosdt']})", "corels": f"CORELS rule lists (λ_reg = {BEST['corels']})"}

rows, lengths = [], {}
for scheme, split_list in SCHEMES.items():
    for split in split_list:
        train, validation, test = sides(split)
        predictors = {NAMES[m]: proba_of(m, split, BEST[m]) for m in ("gosdt", "corels")}
        for m in ("gosdt", "corels"):
            lengths.setdefault(NAMES[m], []).append(size_of(m, split, BEST[m])["numbers to write it down"])
        if ("4-class", split) not in forest_probas:
            _, forest_probas["4-class", split] = forest_4class(train)
        predictors["random forest, 4 classes"] = forest_probas["4-class", split]
        for name, proba in predictors.items():
            lam = calibrate_on(proba, validation)
            for subset, mask in SUBSETS.items():
                rows.append({"predictor": name, "scheme": scheme, "split": split, "subset": subset, "λ": lam, **evaluate(proba, test & mask, lam)})
held_out = pd.DataFrame(rows)
PREDICTORS = list(dict.fromkeys(held_out["predictor"]))

tables, means = {}, {}
for name in PREDICTORS:
    blocks = {}
    for scheme, split_list in SCHEMES.items():
        d = held_out[(held_out["predictor"] == name) & (held_out["scheme"] == scheme)]
        by = d.groupby("subset", sort=False)[list(METRICS)]
        if scheme == "learning to split":
            spread = by.std()
            blocks[f"{scheme} (mean ± SD)"] = formatted(by.mean()) + " ± " + formatted(spread)
        else:
            spread = by.apply(lambda g: g.apply(corrected_se, n_splits=len(split_list), test_to_train=test_to_train(split_list)))
            blocks[f"{scheme} (mean ± SE)"] = formatted(by.mean()) + " ± " + formatted(spread)
        for subset in SUBSETS:
            means[name, scheme, subset] = (by.mean().loc[subset, "information (bits)"], spread.loc[subset, "information (bits)"])
    tables[name] = pd.concat(blocks, names=["split", "subset"])
pd.concat(tables, names=["predictor"])[["information (bits)", "accuracy: same tier (%)", "1 tier off (%)", "2+ tiers off (%)"]]

.005  gosdt  2 leaves   1.3s | corels  3 rules   1.5s | 95s
cluster_6|bit_2|0.005  gosdt  5 leaves   0.8s | corels  4 rules   1.6s | 99s
cluster_7|bit_1|0.005  gosdt  2 leaves   1.2s | corels  3 rules   1.5s | 103s
cluster_7|bit_2|0.005  gosdt  5 leaves   6.2s | corels  3 rules   1.6s | 112s
cluster_8|bit_1|0.005  gosdt  2 leaves   0.6s | corels  3 rules   1.5s | 115s
cluster_8|bit_2|0.005  gosdt  5 leaves   5.6s | corels  3 rules   1.6s | 124s
cluster_9|bit_1|0.005  gosdt  2 leaves   2.1s | corels  3 rules   1.6s | 128s
cluster_9|bit_2|0.005  gosdt  6 leaves   4.0s | corels  3 rules   1.6s | 135s
ls_0|bit_1|0.005       gosdt  4 leaves   0.0s | corels  3 rules   0.1s | 137s
ls_0|bit_2|0.005       gosdt  5 leaves   8.2s | corels  3 rules   1.5s | 147s
ls_1|bit_1|0.005       gosdt  2 leaves   0.0s | corels  3 rules   0.0s | 149s
ls_1|bit_2|0.005       gosdt  5 leaves   1.7s | corels  3 rules   1.5s | 153s
ls_2|bit_1|0.005       gosdt  4 leaves   0.0s | corels  3 rules   0.0s | 155s
ls_2|

information (bits)  \
predictor                         split                                subset                                                      
GOSDT trees (λ_reg = 0.005)       random cross-validation (mean ± SE)  all test compounds                          0.561 ± 0.009   
                                                                       test compounds with 2+ unflagged votes      0.563 ± 0.056   
                                  cluster cross-validation (mean ± SE) all test compounds                          0.546 ± 0.032   
                                                                       test compounds with 2+ unflagged votes      0.546 ± 0.044   
                                  learning to split (mean ± SD)        all test compounds                         -0.753 ± 0.129   
                                                                       test compounds with 2+ unflagged votes     -0.544 ± 0.114   
CORELS rule lists (λ_reg = 0.005) random cross-validation (mean ± SE)  all test compounds                          0.544 ± 0.012   
                                                                       test compounds with 2+ unflagged votes      0.545 ± 0.043   
                                  cluster cross-validation (mean ± SE) all test compounds                          0.543 ± 0.038   
                                                                       test compounds with 2+ unflagged votes      0.539 ± 0.046   
                                  learning to split (mean ± SD)        all test compounds                         -0.732 ± 0.067   
                                                                       test compounds with 2+ unflagged votes     -0.523 ± 0.084   
random forest, 4 classes          random cross-validation (mean ± SE)  all test compounds                          1.009 ± 0.009   
                                                                       test compounds with 2+ unflagged votes      0.991 ± 0.041   
                                  cluster cross-validation (mean ± SE) all test compounds                          0.941 ± 0.025   
                                                                       test compounds with 2+ unflagged votes      0.944 ± 0.047   
                                  learning to split (mean ± SD)        all test compounds                         -0.609 ± 0.117   
                                                                       test compounds with 2+ unflagged votes     -0.273 ± 0.086   

                                                                                                              accuracy: same tier (%)  \
predictor                         split                                subset                                                           
GOSDT trees (λ_reg = 0.005)       random cross-validation (mean ± SE)  all test compounds                                  64.4 ± 0.5   
                                                                       test compounds with 2+ unflagged votes              64.4 ± 2.0   
                                  cluster cross-validation (mean ± SE) all test compounds                                  63.8 ± 1.1   
                                                                       test compounds with 2+ unflagged votes              63.9 ± 2.2   
                                  learning to split (mean ± SD)        all test compounds                                  26.6 ± 2.1   
                                                                       test compounds with 2+ unflagged votes              35.5 ± 1.4   
CORELS rule lists (λ_reg = 0.005) random cross-validation (mean ± SE)  all test compounds                                  63.7 ± 0.7   
                                                                       test compounds with 2+ unflagged votes              63.8 ± 1.6   
                                  cluster cross-validation (mean ± SE) all test compounds          

**Reading the table.** The optimal trees carry 0.561 ± 0.009 bits on random splits and 0.546 on cluster splits; the rule lists 0.544 and 0.543. Against the forest on the same descriptors (1.009 and 0.941) that is a little over half the information, for about twenty numbers instead of three million — and the two-bit factorization is not what is missing: two per-bit forests multiplied lose only 0.017 bits to the four-class forest on validation. What is missing is depth. The regularization grid chose its smallest value for both solvers, and the step from 0.02 to 0.005 bought 0.02 bits for two more leaves; the solvers' own size penalty, not the search, is what stops them at eight leaves, and the validation curve is still rising there.

Across schemes the picture is flat. The trees lose 0.015 bits from random to cluster splits and the rule lists nothing, where the forest loses 0.07: a model that asks three questions about logP is not learning chemistry that fails to transfer. On the learned splits both are between the forest (−0.61) and the additive models' territory at about −0.75, with the same accuracy profile as every model there (26% same tier, 60% one off). On the benchmark population both sit at 0.56 bits with the large standard error that population carries.

Hard-tier accuracies say the same thing from the other side: 64.4% and 63.7% in the same tier against the forest's 73.7%, and 4.7% two or more tiers off against 2.6%. The trees and rule lists make the same errors as each other; the two solvers agree to within one standard error on every scheme, as two provably optimal models of the same size on the same binary features should.

## The models themselves

The chosen trees and rule lists for the first random split, both bits. Then, across the ten random splits, how often each descriptor appears in a tree or a rule list: a descriptor that every split uses is a property of the data, one used once is a property of the fold. Then the description lengths, written with the held-out information to `data/aqsoldb_front_trees_and_rules.csv`.

In [5]:
import re

for m in ("gosdt", "corels"):
    for bit in GRAY:
        print(f"===== {NAMES[m]}, {bit} (1 = tier in {GRAY[bit]}), split {CV_SPLITS[0]} =====")
        print(models[f"{m}|{CV_SPLITS[0]}|{bit}|{BEST[m]}"]["text"])
        print()

rows = []
for m in ("gosdt", "corels"):
    for bit in GRAY:
        used = [set(re.findall(r"([A-Za-z_][A-Za-z0-9_]*) <= ", models[f"{m}|{s}|{bit}|{BEST[m]}"]["text"])) for s in CV_SPLITS]
        counts = pd.Series([d for u in used for d in u]).value_counts()
        for d, n in counts.items():
            rows.append({"solver": m, "bit": bit, "descriptor": d, "splits using it (of 10)": n})
usage = pd.DataFrame(rows).pivot_table(index="descriptor", columns=["solver", "bit"], values="splits using it (of 10)", fill_value=0).astype(int)
usage["total"] = usage.sum(axis=1)
usage = usage.sort_values("total", ascending=False).drop(columns="total")
display(usage.head(20))

length = pd.Series({name: np.mean(v) for name, v in lengths.items()})
length["random forest, 4 classes"] = np.mean([sum(t.tree_.n_leaves for t in forest_4class(sides(s)[0])[0].estimators_) * TIERS for s in CV_SPLITS[:1]])
front = pd.DataFrame(
    [{"model": name, "family": "random forest" if name.startswith("random forest") else ("tree" if name.startswith("GOSDT") else "rule list"),
      "inputs": f"RDKit, one per group ({X.shape[1]})", "scheme": scheme, "subset": subset,
      "information (bits)": means[name, scheme, subset][0], "spread": means[name, scheme, subset][1],
      "numbers to write it down": length[name], "readable": not name.startswith("random forest")}
     for name in PREDICTORS for scheme in SCHEMES for subset in SUBSETS]
)
front.to_csv(ROOT / "data" / "aqsoldb_front_trees_and_rules.csv", index=False)
sizes = pd.DataFrame({NAMES[m]: pd.DataFrame([size_of(m, s, BEST[m]) for s in CV_SPLITS]).mean() for m in ("gosdt", "corels")}).T
sizes.round(1)

===== GOSDT trees (λ_reg = 0.005), bit_1 (1 = tier in (2, 3)), split cv_0 =====
if MolLogP <= 2.756:
  -> 1
else:
  -> 0

===== GOSDT trees (λ_reg = 0.005), bit_2 (1 = tier in (1, 2)), split cv_0 =====
if MolLogP <= 1.578:
  if BertzCT <= 233.3:
    -> 0
  else:
    if MolLogP <= 0.8079:
      if FpDensityMorgan3 <= 2.172:
        -> 0
      else:
        -> 1
    else:
      -> 1
else:
  if MolLogP <= 4.53:
    -> 1
  else:
    -> 0

===== CORELS rule lists (λ_reg = 0.005), bit_1 (1 = tier in (2, 3)), split cv_0 =====
RULELIST:
if [MolLogP <= 2.756 && not SPS <= 2.75]:
  prediction = True
else 
  prediction = False

===== CORELS rule lists (λ_reg = 0.005), bit_2 (1 = tier in (1, 2)), split cv_0 =====
RULELIST:
if [qed <= 0.7377 && not MolLogP <= 4.386]:
  prediction = False
else if [MolLogP <= 1.578 && BertzCT <= 233.3]:
  prediction = False
else 
  prediction = True



solver            corels       gosdt      
bit                bit_1 bit_2 bit_1 bit_2
descriptor                                
MolLogP               10    10    10    10
BertzCT                8     9     0     9
MinPartialCharge       6     0     0     0
qed                    0     3     0     3
ExactMolWt             0     1     0     1
FpDensityMorgan3       0     0     0     2
SPS                    2     0     0     0
Chi1                   0     0     0     1
Chi3v                  0     0     0     1
MaxAbsEStateIndex      1     0     0     0
SMR_VSA7               0     0     0     1

,conditions,descriptors used,leaves (both trees),max depth,numbers to write it down,rules (both lists)
GOSDT trees (λ_reg = 0.005),NaN,5.7,7.7,3.7,19.1,NaN
CORELS rule lists (λ_reg = 0.005),7.7,NaN,NaN,NaN,21.3,5.9


## Reading

1. **How many bits does a provably best small tree carry?** About 0.56 — 0.561 ± 0.009 bits for a pair of optimal trees with eight leaves between them, 0.544 for a pair of rule lists with six rules — against 1.009 for the forest on the same 152 descriptors. The two-bit factorization costs 0.017 bits (measured with the forest, before any tree was fitted), so the gap is the size: both solvers chose the loosest regularization on offer and were still gaining as they grew. A provably best small tree is not a better small tree by much; the sweep's three settings span 0.53–0.56 bits, and the same information is available from a single greedy split (the high-bit tree *is* a single split).

2. **What does it ask?** One question decides the soluble side: in all ten random splits the high-bit tree is the single split *MolLogP ≤ 2.76*, two leaves, nothing else — and the rule list for the same bit is the same question with one qualifier (*and SPS > 2.75*, which excludes flat aromatic frames; in six splits the qualifier is a partial-charge condition instead). The middle bit needs more: the low-bit tree splits on MolLogP at about 1.6 and 4.5 with BertzCT (molecular complexity) in nine of ten splits and qed in three, which reads as "between logP 1.6 and 4.5 you are in the middle tiers unless the molecule is small and simple, in which case it is on the soluble end". The descriptors the trees never ask about are as telling: no size term beyond BertzCT, no polar surface area, no hydrogen-bond counts — with eight leaves to spend, logP and one complexity measure are all that pay.

3. **Where the first question lands.** The threshold logP 2.76 for the soluble side is the same in every split to three figures because the binarizer proposes it and the solver, each time, keeps it. The additive model's logP curve put its steepest fall between logP 2 and 4; the tree puts its one cut in the middle of that range. Two families with nothing in common but the data agree on where solubility turns.

4. **On the front.** Twenty numbers for 0.56 bits puts the trees and rule lists at the same length as the equations and about 0.06 bits below them; the front notebook draws the comparison. The length is honest in a way the EBM's is not — a reader takes in all twenty — and the solvers' size penalty, not a reader's patience, is what the next bits would cost.